# 14. How much each protection matters (the leakage ladder)

The ridge models of step 7 (TF-IDF, Qwen3-Embedding-8B, Mistral-Small-24B) under weaker protocols, always tested on
the **same** 25,524 articles of the 226 outlets of the paper, in the same folds:

| rung | training outlets and articles | text | compared with |
|---|---|---|---|
| article split | the 226 outlets, folds of articles (an outlet is on both sides) | anonymous | L4 |
| sisters | the 226 + always the sisters (outlets of the same owner), all their articles, copies included | original | L2 |
| L2 | the 226 (one outlet per ownership group) | original | L4 |
| L3 | the 226 | + links, e-mails, the outlet's own name | L4 |
| L4 | the 226 | + GLiNER (the paper, step 7) | |

The sisters are put in training on purpose, in every fold: every paper outlet of their group is tested with a sister in
training. Being a sister is only a hint that the texts are similar, so the similarity of test and training articles is
measured too (TF-IDF cosine, near copies), with how much the NewsGuard score changes inside an ownership group. The
sisters come back from the step-2 cache with the same functions and constants (checked against the paper's articles);
details in `ladder.py`. Order of the work: the stages, TF-IDF on every rung, the frozen models on the article split
(vectors of step 5), then the new vectors (rules text: 226 outlets; original text: the paper's and the sisters'
articles) on the GPU and the frozen models on the other rungs.

In [1]:
N_JOBS = 16
GPU_GB = 19
QWEN_BUDGET = 16384
MISTRAL_BUDGET = 8192

In [2]:
# Parameters
N_JOBS = 8


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import ladder as LA
import paper as PA

A, by_step = LA.stages(N_JOBS, LOG)
sis, with_sister = LA.sisters(A)
print(f"paper: {A[A.s2].domain.nunique()} outlets, {A.s2.sum():,} articles | sisters: {len(sis)} outlets, "
      f"{A.domain.isin(sis).sum():,} articles | paper outlets with a sister: {len(with_sister)}")
by_step["last step"].value_counts()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


paper: 226 outlets, 25,524 articles | sisters: 89 outlets, 12,165 articles | paper outlets with a sister: 31


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


last step
sent to the scraper     619
one outlet per group    226
from 2025               170
text extracted           91
>= 10 articles           61
no near duplicates       60
100-1,500 words          34
no exact duplicates      14
English                  12
at least one article      6
Name: count, dtype: int64

## TF-IDF on every rung, the frozen models on the article split

In [4]:
for rung in LA.RUNGS:
    LA.run(rung, A, ["TF-IDF"], N_JOBS, LOG)
LA.run("article split", A, ["Qwen3-8B", "Mistral-24B"], N_JOBS, LOG)

2026-11-17 10:59:14  ladder article split: 226 outlets, 25,524 articles, anonymous text


2026-11-17 10:59:14  regression, TF-IDF, fold 1: already done


2026-11-17 10:59:14  regression, TF-IDF, fold 2: already done


2026-11-17 10:59:14  regression, TF-IDF, fold 3: already done


2026-11-17 10:59:14  regression, TF-IDF, fold 4: already done


2026-11-17 10:59:14  regression, TF-IDF, fold 5: already done


... (25 outputs left out)

2026-11-17 11:30:58  regression, Mistral-24B, fold 2: already done


2026-11-17 11:30:58  regression, Mistral-24B, fold 3: already done


2026-11-17 11:30:58  regression, Mistral-24B, fold 4: already done


2026-11-17 11:30:58  regression, Mistral-24B, fold 5: already done


## New frozen vectors (GPU) and the frozen models on the other rungs

In [5]:
LA.embed("rules", A, LOG, GPU_GB, QWEN_BUDGET, MISTRAL_BUDGET)
LA.run("L3", A, ["Qwen3-8B", "Mistral-24B"], N_JOBS, LOG)
LA.embed("original", A, LOG, GPU_GB, QWEN_BUDGET, MISTRAL_BUDGET)
for rung in ["sisters", "L2"]:
    LA.run(rung, A, ["Qwen3-8B", "Mistral-24B"], N_JOBS, LOG)

2026-11-17 11:30:59  ladder L3: 226 outlets, 25,524 articles, rules text


2026-11-17 11:30:59  regression, Qwen3-8B, fold 1: already done


2026-11-17 11:30:59  regression, Qwen3-8B, fold 2: already done


2026-11-17 11:30:59  regression, Qwen3-8B, fold 3: already done


2026-11-17 11:30:59  regression, Qwen3-8B, fold 4: already done


2026-11-17 11:30:59  regression, Qwen3-8B, fold 5: already done


... (33 outputs left out)

2026-11-17 11:36:40  regression, Mistral-24B, fold 2: already done


2026-11-17 11:36:40  regression, Mistral-24B, fold 3: already done


2026-11-17 11:36:40  regression, Mistral-24B, fold 4: already done


2026-11-17 11:36:40  regression, Mistral-24B, fold 5: already done


## Results (the same outlets and test articles for every rung)

In [6]:
art, outlets, rows = PA.load()
L = PA.ladder_results(art, outlets)
L.to_csv(C.OUT / "paper_ladder.csv", index=False)
L.assign(text=[f"{v:.3f}" if m == "Spearman" else f"{v:.1f}" for v, m in zip(L.value, L.measure)]) \
    .pivot_table(index=["outlets", "rung"], columns=["measure", "model"], values="text", aggfunc="first")

measure                                MAE                    Spearman                
model                          Mistral-24B Qwen3-8B TF-IDF Mistral-24B Qwen3-8B TF-IDF
outlets          rung                                                                 
all              L2                   15.7     16.0   18.0       0.734    0.724  0.673
                 L3                   15.7     15.9   17.7       0.734    0.724  0.679
                 L4                   15.8     16.3   17.8       0.735    0.711  0.674
                 article split         8.7     10.7   10.4       0.924    0.884  0.904
                 sisters              14.9     15.7   17.1       0.732    0.733  0.712
with a sister    L2                   18.3     18.4   19.9       0.732    0.681  0.664
                 sisters               9.7     11.6    8.3       0.943    0.908  0.918
without a sister L2                   15.3     15.6   17.7       0.739    0.734  0.667
                 sisters              15.8     16.3   18.5       0.698    0.707  0.676

In [7]:
L[L.p.notna() & (L.measure == "MAE")][["outlets", "n", "rung", "model", "diff", "dlo", "dhi", "p"]].round(3)

,outlets,n,rung,model,diff,dlo,dhi,p
1,all,226,article split,TF-IDF,-7.435,-8.513,-6.418,0.000
3,all,226,article split,Qwen3-8B,-5.620,-6.458,-4.819,0.000
5,all,226,article split,Mistral-24B,-7.142,-8.190,-6.142,0.000
7,all,226,sisters,TF-IDF,-0.837,-1.867,0.085,0.079
9,all,226,sisters,Qwen3-8B,-0.333,-1.151,0.457,0.436
11,all,226,sisters,Mistral-24B,-0.808,-1.764,0.129,0.095
13,all,226,L2,TF-IDF,0.138,-0.024,0.306,0.093
15,all,226,L2,Qwen3-8B,-0.327,-0.688,0.025,0.074
17,all,226,L2,Mistral-24B,-0.060,-0.214,0.094,0.441
19,all,226,L3,TF-IDF,-0.126,-0.293,0.046,0.149


## Similarity of test and training articles, and the NewsGuard score inside an ownership group

For every test article, the highest TF-IDF cosine with a training article without the sisters and with them, and
whether a sister has a near copy of it; then, per outlet, whether the drop of the error follows the similarity.

In [8]:
LA.similarity(A, N_JOBS, LOG)
T, R = PA.sister_similarity(art, outlets)
T.to_csv(C.OUT / "paper_sisters_similarity.csv", index=False)
R.to_csv(C.OUT / "paper_sisters_gain.csv", index=False)
display(T.round(3)); display(R.round(3))

2026-11-17 11:38:43  similarity: fold 1 done


2026-11-17 11:39:15  similarity: fold 2 done


2026-11-17 11:39:51  similarity: fold 3 done


2026-11-17 11:40:29  similarity: fold 4 done


2026-11-17 11:41:13  similarity: fold 5 done


,outlets,n,similarity without the sisters,with the sisters,% closest training article is a sister's,% near copy in a sister
0,with a sister,31,0.185,0.603,69.634,49.808
1,without a sister,195,0.163,0.176,25.086,0.559


,model,outlets,n,Spearman,low,high
0,TF-IDF,all,226,0.267,0.133,0.391
1,TF-IDF,with a sister,31,0.599,0.301,0.804
2,Qwen3-8B,all,226,0.116,-0.023,0.251
3,Qwen3-8B,with a sister,31,0.247,-0.108,0.570
4,Mistral-24B,all,226,0.194,0.055,0.323
5,Mistral-24B,with a sister,31,0.319,-0.028,0.623


In [9]:
G = PA.owner_scores()
G.to_csv(C.OUT / "paper_owner_scores.csv", index=False)
G.round(1)

,pairs,outlets,mean |difference|,median |difference|,% the same score
0,the same owner (paper outlet and its sisters),31,3.8,0.0,74.2
1,any two of these outlets,120,29.6,25.0,6.7
